In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv('/home/jsoychak/cs/ecg/data/mitbih_train.csv')
df.describe()

In [ ]:
data = df.to_numpy()
X = data[:-1]
y = data[-1]

fig, axes = plt.subplots(5, 1)
freq = 125 # Hz
t = np.linspace(0, len(X[0])/freq, len(X[0]))
for ax, x in zip(axes, X[:5]):
    ax.plot(t, x, color='black', linewidth=1)
plt.show()

In [ ]:
import copy
import keras

class EcgDataset(keras.utils.PyDataset):
    def __init__(self, X, batch_size, wave_size, window_size, noise_std=0.05, **kwargs):
        super().__init__(**kwargs)
        self.X = X
        self.batch_size = batch_size
        self.wave_size = wave_size
        self.window_size = window_size
        self.noise_std = noise_std

        self.X_noisy = copy.deepcopy(X)
        self.X_noisy += np.random.normal(loc=0, scale=noise_std, size=self.X_noisy.shape)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        assert len(self.X[idx]) >= self.window_size
        step_size = (len(self.X[idx]) - self.window_size + 1)//self.batch_size
        steps = range(0, step_size*self.batch_size, step_size)

        x_batch = np.array([
            self.X_noisy[idx][s: s + self.window_size] for s in steps
        ])

        y_batch = np.array([
            self.X[idx][s + (self.window_size - self.wave_size)//2:
                   s + (self.window_size - self.wave_size)//2 + self.wave_size]
            for s in steps
        ])
        return x_batch, y_batch

In [ ]:
X = np.expand_dims(X, axis=-1)

In [ ]:
batch_size = 16
wave_size = 64
window_size =  96
dataset = EcgDataset(X, batch_size, wave_size, window_size)
x, y = dataset.__getitem__(0)
print(f'x.shape: {x.shape} y.shape: {y.shape}')

In [ ]:
def plot_batch(x, y):
    x = np.squeeze(x)
    y = np.squeeze(y)

    batch_size, window_size = x.shape
    _, wave_size = y.shape

    _, axes = plt.subplots(batch_size, 2, figsize=(10, batch_size * 2))
    axes = np.atleast_2d(axes)

    batch = zip(x, y)
    for (x_i, y_i), (ax_x, ay_y) in zip(batch, axes):
        ax_x.plot(np.arange(window_size), x_i)
        ay_y.plot(np.arange(wave_size), y_i)
    
    plt.tight_layout()
    plt.show()

plot_batch(x, y)

In [ ]:
import tensorflow as tf

class WaveTransformer(tf.keras.Model):
    def __init__(self, window_size, wave_size, hidden_units, stride=1, **kwargs):
        super().__init__(**kwargs)
        self.window_size = window_size
        self.wave_size = wave_size
        self.stride = stride

        # waves
        self.waves = tf.Variable(tf.random.normal((wave_size, wave_size)), trainable=True)

        # wave space neural network
        self.nn = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_units, activation='relu'),
            tf.keras.layers.Dense(wave_size)
        ])
    
    def call(self, x):
        # orthogonal componets
        Q, _ = tf.linalg.qr(self.waves)
        
        # transform into the wave space
        kernel = tf.reshape(Q, shape=(self.wave_size, 1, self.wave_size))
        z = tf.nn.conv1d(x, filters=kernel, stride=self.stride, padding='VALID')

        # average wave amplitudes
        z = tf.reduce_mean(z, axis=1)

        # reconstruct the wave from the excited wave amplitudes
        z = self.nn(z)

        # transform into the time space
        z = tf.matmul(z, tf.transpose(Q))

        return tf.expand_dims(z, axis=-1)
    
    def wave_transform(self, x, pooling='avg'):
        x = tf.cast(x, dtype=tf.float32)

        Q, _ = tf.linalg.qr(self.waves)

        # transform into the wave space
        kernel = tf.reshape(Q, shape=(self.wave_size, 1, self.wave_size))
        z = tf.nn.conv1d(x, filters=kernel, stride=self.stride, padding='VALID')

        if (pooling == 'avg'):
            z = tf.reduce_mean(z, axis=1)

        return z
    
    def wave_transform_inverse(self, x):
        x = tf.cast(x, dtype=tf.float32)

        Q, _ = tf.linalg.qr(self.waves)

        return tf.matmul(x, tf.transpose(Q))


In [ ]:
hidden_units = 256
model = WaveTransformer(window_size, wave_size, hidden_units)
model.compile(
    optimizer=tf.optimizers.Adam(),
    loss=tf.keras.losses.MeanSquaredError()
)

In [ ]:
epochs = 10
history = model.fit(dataset, epochs=epochs)

In [ ]:
y_pred = model(x)
y_true = y

In [ ]:
x_tensor = tf.cast(x, tf.float32)
model.wave_transform(x_tensor)

In [ ]:
def plot_batch_results(model, x, y_true):
    y_pred = model(x)
    x = np.squeeze(x)
    y_true = np.squeeze(y_true)
    y_pred = np.squeeze(y_pred)
    results = zip(x, y_pred, y_true)

    batch_size, window_size = x.shape
    _, wave_size = y_true.shape

    _, axes = plt.subplots(batch_size, 3, figsize=(9, batch_size))
    for (x_i, y_pred_i, y_true_i), (ax_i, ay_pred_i, ay_true_i) in zip(results, axes):
        ax_i.plot(np.arange(window_size), x_i)
        ay_pred_i.plot(np.arange(wave_size), y_pred_i)
        ay_true_i.plot(np.arange(wave_size), y_true_i)

    plt.tight_layout()
    plt.show()

plot_batch_results(model, x, y)

In [ ]:
def plot_batch_wave_transform(model, x, y_true):
    x = tf.cast(x, dtype=tf.float32)
    y_pred = model(x)
    y_true = tf.cast(y_true, dtype=tf.float32)

    x_amp = model.wave_transform(x)
    y_pred_amp = model.wave_transform(y_pred)
    y_true_amp = model.wave_transform(y_true)

    amps = zip(x_amp, y_pred_amp, y_true_amp)

    batch_size, wave_size = x_amp.shape
    _, axes = plt.subplots(batch_size, 3, figsize=(9, 2*batch_size))

    for (x_i, y_pred_i, y_true_i), (ax_i, ay_pred_i, ay_true_i) in zip(amps, axes):
        ax_i.scatter(np.arange(wave_size), x_i)
        ay_pred_i.plot(np.arange(wave_size), y_pred_i)
        ay_true_i.plot(np.arange(wave_size), y_true_i)

    plt.tight_layout()
    plt.show()

plot_batch_wave_transform(model, x[:4], y[:4])

In [ ]:
random_model = WaveTransformer(window_size, wave_size, hidden_units)

In [ ]:
plot_batch_wave_transform(random_model, x[:4], y[:4])